## 1. Import File Path and Read PDFs

In [39]:
import os
import pdfplumber

import pandas as pd

folder_path = "../data/raw/japan_pdfs"
os.listdir(folder_path)

['71220_196311.pdf',
 '71621_198006.pdf',
 '71320_200509.pdf',
 '71620_196912.pdf',
 '71220_201412.pdf',
 '71320_201212.pdf',
 '71111_201412.pdf',
 '71620_197212.pdf',
 '71620_200908.pdf',
 '71650_201212.pdf',
 '71220_201212.pdf',
 '71621_197910.pdf',
 '71220_200509.pdf',
 '71620_197612.pdf',
 '71320_196311.pdf',
 '71322_197910.pdf',
 '71320_201412.pdf',
 '71620_200311.pdf',
 '71440_201710.pdf',
 '71320_198006.pdf',
 '71321_196912.pdf',
 '71660_201412.pdf',
 '71530_200311.pdf',
 '71220_196011.pdf',
 '71220_197910.pdf',
 '71220_201710.pdf',
 '71321_197212.pdf',
 '71630_201212.pdf',
 '71624_201412.pdf',
 '71630_200509.pdf',
 '71530_197612.pdf',
 '71320_196011.pdf',
 '71530_200908.pdf',
 '71530_197212.pdf',
 '71320_197910.pdf',
 '71321_197612.pdf',
 '71220_198006.pdf',
 '71530_196912.pdf',
 '71624_200509.pdf',
 '71410_201212.pdf',
 '71320_201710.pdf',
 '71640_200311.pdf',
 '71624_201212.pdf',
 '71620_197910.pdf',
 '71624_200908.pdf',
 '71620_196011.pdf',
 '71621_197612.pdf',
 '71410_20090

In [53]:
def read_pdf(file_path):
    result = {}
    with pdfplumber.open(file_path, laparams={"detect_vertical": True}) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text(x_tolerance=6, y_tolerance=40)
            result[f"Page {page_num}"] = text if text else ""
    return result

def read_pdf_folder_df(folder_path):
    rows = []
    for filename in sorted(os.listdir(folder_path)):
        if not filename.lower().endswith(".pdf"):
            continue
        file_path = os.path.join(folder_path, filename)
        try:
            pages = read_pdf(file_path)
            full_text = "\n".join(pages.values())
            rows.append({"filename": filename, "text": full_text, "n_pages": len(pages)})
        except Exception as e:
            rows.append({"filename": filename, "text": None, "n_pages": None})
            print(f"Failed on {filename}: {e}")
    return pd.DataFrame(rows)

with pdfplumber.open("../data/raw/japan_pdfs/71220_201412.pdf", laparams={"detect_vertical": True}) as pdf:
    words = pdf.pages[0].extract_words(x_tolerance=6, y_tolerance=40)
    for w in words[:15]:
        print(w["text"], w.get("upright"))

新 True
しい True
政 True
治 True
を True
国 True
民 True
の True
声 True
が True
生 True
き True
！ True
安倍政権の暴走ストップ True
る True


## 2. Merge with sorted metadata text csv

In [54]:
japan_texts_df = read_pdf_folder_df(folder_path)

sorted_manifestos = pd.read_csv("../data/clean/Japan_sorted_manifestos.csv")
sorted_manifestos["filename"] = sorted_manifestos["manifesto_id"] + ".pdf"

merged = sorted_manifestos.merge(
    japan_texts_df, on="filename", how="left", suffixes=("_meta", "_pdf")
)

merged["text_available"] = merged["text_pdf"].apply(
    lambda t: isinstance(t, str) and t.strip(" \n") != ""
)

print(merged.loc[~merged["text_available"], ["filename", "partyname", "manifesto_id"]])

           filename                                 partyname  manifesto_id
5  71440_201710.pdf  Constitutional Democratic Party of Japan  71440_201710


In [55]:
merged.head(10)

,date,partyname,manifesto_id,text_meta,filename,text_pdf,n_pages,text_available
0,2014-12,Japanese Communist Party,71220_201412,安倍政権の暴走ストップ！ 国民の声が生きる新しい政治を 日本共産党の総選挙政策 日本共産党 ...,71220_201412.pdf,新 しい\n政\n治\nを\n国 民 の\n声\nが\n生 き ！\n安倍政権の暴走ストップ...,20,True
1,2014-12,Social Democratic Party,71320_201412,約束１ アベノミクスによる生活破壊を許さず、拡大した格差を是正します （１）景気を悪化させる...,71320_201412.pdf,衆議院選挙公約\n2014\n約 約 約約約約約約 １１ 44 2233 束束束束束束 束束...,8,True
2,2017-10,Japanese Communist Party,71220_201710,日本共産党の総選挙政策 日本共産党 安倍首相は、臨時国会の冒頭解散に打って出ました。 「森友...,71220_201710.pdf,印第頒刷4布8者責衆／任 東者日京／本都東共新京産宿都党区渋法築谷定地区パ町千ン8駄フ－ヶレ...,24,True
3,2017-10,Social Democratic Party,71320_201710,くらし支えます 1家計を温めボトムアップの経済政策でくらしの再建 憲法１３条の幸福追求権、２...,71320_201710.pdf,ア優 社ベ民 先 党政の は 「治約 憲 束のし 法 ま暴す を走活スかトッすプ政、治国」を...,8,True
4,2017-10,Japan Restoration Party,71430_201710,消費増税凍結! 維新ならできる! 増税なしで改革実現! 身を切る改革で財源を生み出し、議員報...,71430_201710.pdf,消身費を増切る税改 革凍で結教！育無償化。\n日本維新の会｜2017維新八策\n消 身 2 ...,13,True
5,2017-10,Constitutional Democratic Party of Japan,71440_201710,1 生活の現場から暮らしを立て直します アベノミクスの成果は上がらず、国民の所得を削り、中間...,71440_201710.pdf,\n\n\n\n\n\n\n,8,False
6,2017-10,New Clean Government Party,71530_201710,教育負担の軽減へ。 衆院選で公明党は、「教育負担の軽減へ。」を掲げます。 国づくりの基...,71530_201710.pdf,衆院選 Manifesto2017\n重\n点\n政\n策\n教育負担の\n軽減へ。\n公明...,13,True
7,2017-10,Liberal Democratic Party,71620_201710,北朝鮮の脅威から、 国民を守り抜きます わが国の上空を飛び越える弾道ミサイルの相次ぐ発...,71620_201710.pdf,守 こ\nり の\n抜 国\nく を\n。、\nあなたの一票を自民党に。\n第48衆 自由民...,20,True
8,2017-10,Party of Hope,71627_201710,私たちが希求するものは、党の利益ではなく、議員の利益でもなく、 国民のため、つまり国民...,71627_201710.pdf,消 「をに で 景めを 現道 徹 競地 2目 決 変 場州 底 凍気、 0原 費 指 め え...,14,True
